# RQ3 — Histogram matching (aussi appliqué à la validation et au test)

Run `unet3d_histogram_matching_pli0` (notebook Kaggle `rq3-histogram-matching-pli0`) du tableau d'ablation : UNet MONAI 3D en fp32 (clipping des
gradients à 12), patches de 96³, **pli 0** de `splits.json`, **graine 2026**, 40 000 itérations (lots de 4 patches),
identiques pour toutes les techniques. Durée estimée sur une T4 : 7 à 7 h 30 (garde-fou à 10 h d'entraînement,
session limitée à 12 h). Si l'itération 100 dépasse 0.75 s/it, un avertissement est écrit dans le journal.

**Avant de lancer** : accélérateur *GPU T4*, Internet activé (installation de MONAI et TorchIO), datasets privés
`atlas-c1-processed` (données C1) et `projet-avc-code` (code) ajoutés en entrée.
Notebook à garder **privé** (accord d'utilisation ATLAS). *Save Version → Save & Run All* pour une exécution en arrière-plan.
Toutes les sorties sont écrites dans `/kaggle/working/runs/unet3d_histogram_matching_pli0/`.

In [ ]:
# Environnement : GPU obligatoire (accélérateur « GPU T4 »), sinon arrêt immédiat avec un message clair.
import hashlib, json, os, re, shutil, subprocess, sys, time, zipfile
from pathlib import Path
import torch
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| GPU",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "aucun")
if not torch.cuda.is_available() and os.environ.get("RQ3_GPU", "1") == "1":
    raise RuntimeError("AUCUN GPU DISPONIBLE : dans le panneau de droite, Settings > Accelerator > « GPU T4 x2 », "
                       "puis relancer (Save Version > Save & Run All). Rien n'a été lancé.")
INPUT = Path(os.environ.get("RQ3_INPUT", "/kaggle/input"))      # datasets attachés
WORKING = Path(os.environ.get("RQ3_WORKING", "/kaggle/working"))  # toutes les sorties
SCRATCH = Path(os.environ.get("RQ3_TMP", "/tmp/rq3"))            # liens vers les données, jamais dans les sorties
EXPECTED_COMMIT = None  # commit attendu dans VERSION.json du dataset de code (None : pas de contrôle)

In [ ]:
# Code du projet : dataset privé projet-avc-code, cherché récursivement sous /kaggle/input.
def code_version(candidate):
    if candidate.suffix == ".zip":
        with zipfile.ZipFile(candidate) as archive:
            return json.loads(archive.read("VERSION.json")) if "VERSION.json" in archive.namelist() else None
    return json.loads(candidate.read_text())

candidates = sorted(INPUT.rglob("projet_avc_code.zip")) + sorted(
    path for path in INPUT.rglob("VERSION.json") if (path.parent / "adaptation" / "train.py").is_file())
found = [(path, code_version(path)) for path in candidates]
found = [(path, version) for path, version in found if version]
if not found:
    raise FileNotFoundError("Dataset projet-avc-code introuvable sous /kaggle/input : Add Input > Datasets > "
                            "« projet-avc-code » (onglet Shared with you), puis relancer.")
matching = [(path, version) for path, version in found if EXPECTED_COMMIT in (None, version["commit"])]
if not matching:
    raise RuntimeError(f"VERSION.json : commit {sorted({v['commit'] for _, v in found})} au lieu de {EXPECTED_COMMIT}. "
                       "Mettre l'entrée projet-avc-code à sa dernière version (menu ⋮ de l'entrée), puis relancer.")
source, VERSION = matching[0]
print("Code :", source, VERSION)
REPO = WORKING / "PROJET_AVC"
if not REPO.exists():
    if source.suffix == ".zip":
        zipfile.ZipFile(source).extractall(REPO)
    else:  # Kaggle a décompressé l'archive : copie de l'arborescence
        shutil.copytree(source.parent, REPO)
if os.environ.get("RQ3_PIP", "1") == "1":
    pip = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "monai==1.6.1", "torchio==1.2.1",
                          "nibabel==5.4.2", "pyarrow"])
    if pip.returncode:
        raise RuntimeError("Installation de MONAI / TorchIO impossible : activer Internet "
                           "(Settings > Internet on, compte vérifié par téléphone), puis relancer.")

In [ ]:
# Données C1 : dataset privé atlas-c1-processed, cherché récursivement ; liens vers les dossiers de sujets.
tables = sorted(INPUT.rglob("subjects.parquet"))
if len(tables) != 1:
    raise FileNotFoundError(f"subjects.parquet trouvé {len(tables)} fois sous /kaggle/input (1 attendu) : "
                            "ajouter le dataset atlas-c1-processed (Add Input > Datasets > Shared with you).")
TABLE = tables[0]
DATA = SCRATCH / "processed"
DATA.mkdir(parents=True, exist_ok=True)
subjects = sorted(INPUT.rglob("qc.json"))
if not subjects:  # archives restées compressées : extraction hors des sorties
    for archive in sorted(INPUT.rglob("processed_partie*.zip")):
        zipfile.ZipFile(archive).extractall(SCRATCH / "c1_extrait")
    subjects = sorted((SCRATCH / "c1_extrait").rglob("qc.json"))
for qc in subjects:
    link = DATA / qc.parent.name
    if not link.exists():
        link.symlink_to(qc.parent, target_is_directory=True)
if not (DATA / "subjects.parquet").exists():
    (DATA / "subjects.parquet").symlink_to(TABLE)
print(len(subjects), "sujets prétraités ; table", TABLE)
if not subjects:
    raise FileNotFoundError("Aucun dossier de sujet (qc.json) dans atlas-c1-processed")
# Même découpage que le code : splits.json du dataset (à côté de subjects.parquet) identique à celui du dépôt.
data_splits = next((folder / "splits.json" for folder in [TABLE.parent, *TABLE.parents]
                    if folder != INPUT and INPUT in folder.parents and (folder / "splits.json").is_file()), None)
if data_splits is None:
    raise FileNotFoundError("splits.json absent du dataset atlas-c1-processed")
digest = lambda path: hashlib.sha256(Path(path).read_bytes()).hexdigest()
assert digest(data_splits) == digest(REPO / "splits.json"), "splits.json du dataset différent de celui du code"

In [ ]:
# Entraînement, validation sur le pli, puis évaluation sur le test ID du pli et le test OOD.
# Journal complet aussi dans /kaggle/working/runs/unet3d_histogram_matching_pli0/entrainement.log.
# Session interrompue : rattacher la sortie précédente, copier last.pt dans /kaggle/working/runs/unet3d_histogram_matching_pli0/ et ajouter "--resume".
def run_logged(command, log_path, slow_seconds=None, check_at=100):
    """Lance `command` depuis REPO, recopie sa sortie dans le notebook et dans `log_path`.

    Si la durée par itération dépasse `slow_seconds` à l'itération `check_at`, l'écrit clairement dans le
    journal, sans arrêter la run (le garde-fou de 10 h de train.py arrête l'entraînement plus tôt).
    """
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with open(log_path, "a", encoding="utf-8") as log:
        def emit(text):
            print(text, end="", flush=True)
            log.write(text)
            log.flush()
        process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                                   bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"})
        for line in process.stdout:
            emit(line)
            match = re.search(r"'progression': (\d+),.*'s_par_iteration': ([\d.]+)", line)
            if slow_seconds and match and int(match[1]) == check_at and float(match[2]) > slow_seconds:
                seconds = float(match[2])
                reached = int((10 * 3600 - 20 * 270) / seconds)  # budget de 10 h moins ~20 validations de 4 min 30
                emit(f"\n{'!' * 100}\nATTENTION : {seconds:.3f} s/it à l'itération {check_at}, au-dessus de "
                     f"{slow_seconds} s/it.\nÀ ce rythme, 40 000 itérations et les validations ne tiennent pas dans "
                     f"le budget de 10 h : le garde-fou arrêtera l'entraînement vers l'itération {min(reached, 40000)}."
                     f"\nLa run continue (aucun arrêt) ; à signaler dans le tableau RQ3.\n{'!' * 100}\n\n")
        process.wait()
    if process.returncode:
        raise RuntimeError(f"{command[3]} terminé avec le code {process.returncode} : voir {log_path}")

RUN = WORKING / "runs" / "unet3d_histogram_matching_pli0"
run_logged([sys.executable, "-u", "-m", "adaptation.train", "--technique", "histogram_matching", "--fold", "0",
            "--seed", "2026", "--data-root", str(DATA), "--output", str(WORKING / "runs"), "--num-workers", "4",
            "--validate-every", "2000", "--log-every", "100"], RUN / "entrainement.log", slow_seconds=0.75)

In [ ]:
# Résumé et fichiers à récupérer (onglet Output) ; aucune image de patient n'est écrite.
for path in sorted(RUN.glob("resume*.json")):
    print(path.name, json.dumps(json.loads(path.read_text()), indent=2, ensure_ascii=False))
for path in sorted(RUN.iterdir()):
    print(f"{path.stat().st_size / 1e6:10.1f} Mo  {path.name}")